# Ch07 RAG & Embeddings — GPU Supplement

This notebook covers advanced RAG patterns and GPU-accelerated embedding models.

**Experiments**:
1. **Embedding model comparison**: CPU-viable vs GPU-beneficial models
2. **Advanced patterns**: Query decomposition, hybrid search (BM25 + dense)
3. **Re-ranking**: Cross-encoder re-scoring for precision

**Requirements**: CUDA GPU (any size), `sentence-transformers`, `rank_bm25`, test corpus

In [ ]:
import torch

if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU detected — run the CPU notebook instead: notebook-exercise.ipynb\n"
        "To provision a GPU machine: see notes/06-ai-infrastructure/ch01-gpu-architecture/"
    )

device = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {device}  |  VRAM: {vram:.1f} GB")
print("✓ GPU available for accelerated embedding generation")

## Experiment — Embedding Model Precision Comparison

Compare retrieval quality between:
- **all-MiniLM-L6-v2**: 384 dimensions, 22M params, CPU-viable
- **bge-large-en-v1.5**: 1024 dimensions, 335M params, GPU-beneficial

**Metric**: Precision@5 on a test corpus with 10 held-out queries.

In [ ]:
# TODO: Implement embedding model precision comparison
#
# 1. Define test corpus (20 documents on ML/AI topics)
# 2. Define 10 test queries with known relevant documents
# 3. For each model (all-MiniLM-L6-v2, bge-large-en-v1.5):
#    a. Embed corpus + queries (use device='cuda')
#    b. For each query, retrieve top-5 by cosine similarity
#    c. Calculate precision@5 (% of top-5 that are actually relevant)
# 4. Print comparison table: model, dimension, precision@5, avg encoding time
#
# Expected: bge-large should have higher precision but slower encoding

## Analysis

**Trade-offs**:
- **all-MiniLM-L6-v2**: Faster, smaller, good baseline precision, CPU-viable
- **bge-large-en-v1.5**: Higher precision, GPU-beneficial, 3-5× slower per doc

**When to choose**:
- Use MiniLM for prototyping, CPU deployments, or latency-critical applications
- Use bge-large when retrieval quality is paramount and GPU is available

## Key Takeaways

**Embedding model trade-offs:**
- **all-MiniLM-L6-v2**: Fast, CPU-viable, good baseline (384d)
- **bge-large-en-v1.5**: Higher precision, GPU-beneficial, 3-5× slower (1024d)

**Advanced patterns:**
- **Query decomposition**: Split complex queries into atomic sub-queries
- **Hybrid search**: Combine BM25 (keyword) + dense (semantic) with RRF
- **Cross-encoder re-ranking**: Slow but accurate final re-scoring

**When to use each:**
- Start with dense retrieval (bi-encoder) for most cases
- Add BM25 hybrid search for exact keyword needs (codes, names, acronyms)
- Add cross-encoder re-ranking when precision is critical and latency allows
- Use query decomposition for multi-part analytical questions

In [ ]:
# TODO: Implement cross-encoder re-ranking
#
# 1. Use bi-encoder (all-MiniLM-L6-v2) to get top-20 candidates
# 2. Load cross-encoder: 'cross-encoder/ms-marco-MiniLM-L-6-v2'
# 3. For each (query, candidate) pair:
#    - Cross-encoder reads both together
#    - Returns relevance score (not separate embeddings)
# 4. Re-rank top-20 by cross-encoder scores
# 5. Take final top-5
#
# Expected: Cross-encoder improves precision at the cost of speed
# (20 forward passes vs 0 for bi-encoder-only)

## Experiment 4 — Cross-Encoder Re-Ranking

**Problem**: Bi-encoder retrieval is fast but approximate (embeds query and doc separately).

**Solution**: Use cross-encoder to re-score top-k results by reading query+doc together (slower but more accurate).

In [ ]:
# TODO: Implement hybrid search with RRF
#
# 1. Install rank_bm25: pip install rank-bm25
# 2. Define test corpus (20 docs about ML/AI)
# 3. Define query with exact term: "What is DiskANN?"
#
# 4. BM25 retrieval:
#    - Tokenize corpus
#    - Create BM25 object
#    - Get top-5 by BM25 scores
#
# 5. Dense retrieval:
#    - Embed corpus and query
#    - Get top-5 by cosine similarity
#
# 6. Reciprocal Rank Fusion (RRF):
#    - For each doc: score = Σ 1/(k + rank_i) for all retrievers
#    - k = 60 (standard RRF constant)
#    - Merge and sort by RRF score
#
# Expected: BM25 catches exact "DiskANN", dense catches semantic variants

## Experiment 3 — Hybrid Search (BM25 + Dense)

**Problem**: Dense retrieval misses exact keyword matches; sparse retrieval misses paraphrases.

**Solution**: Combine BM25 (sparse/keyword) with dense semantic search using Reciprocal Rank Fusion (RRF).

In [ ]:
# TODO: Implement query decomposition
#
# 1. Define complex query: "Compare the precision of all-MiniLM-L6-v2 and bge-large-en-v1.5 for semantic search"
# 2. Use LLM to decompose into sub-queries:
#    - "What is all-MiniLM-L6-v2 precision?"
#    - "What is bge-large-en-v1.5 precision?"
#    - "How do they compare?"
# 3. Retrieve top-3 for each sub-query independently
# 4. Merge retrieved chunks (deduplicate)
# 5. Generate final answer using all context
#
# Expected: Better retrieval than single complex query

## Experiment 2 — Query Decomposition

**Problem**: Complex multi-part queries produce muddled embeddings that retrieve mediocre results for each sub-question.

**Solution**: Decompose complex queries into atomic sub-queries, retrieve for each, then synthesize.